Setup and imports

In [31]:
import cobra
import csv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [32]:
model = cobra.io.load_json_model("e_coli_core-1.json")

1а) No, as the maximal reaction activities (unlike fluxes) can be unequal along the linear pathway
In 2 consequent reactions flux of the first reaction would be a bottleneck, therefore the second can have the same flux as the first reaction
Maximal reaction activity determines the maximum of activity that potentially a reaction can have, that is why 2 consequent reactions can have different maximums
For example, PFK (13.1) != FBA (30.4) != GAPD (24.5) are not equal along the pathway
1b) Two types of values that can be observed on reactions with grey arrows: "0.00"(for example, PFL, LDH_D) and "no data"(for example, FORti, H2Ot).
"0.00" - the maximal reaction activity was measured and it is equal to 0.00 (numerical value exists)
"no data" - the maximal reaction activity is not determined

In [33]:
Task 2

2.0

In [34]:
activity = {}

filename = "KEN3170_Assignment_2026_e_coli_core_expression.csv"

with open(filename, mode="r", newline="") as file:
    reader = csv.reader(file)
    header = next(reader)

    for row in reader:
        reaction_id = row[0].strip()
        value = float(row[1])
        activity[reaction_id] = value

print("Number of reactions with activity data:", len(activity))

Number of reactions with activity data: 65


In [35]:
list(activity.items())[:10]

[('PFK', 13.1),
 ('PFL', 0.0),
 ('PGI', 11.1),
 ('PGK', 24.0),
 ('PGL', 7.3),
 ('ACALD', 0.0),
 ('AKGt2r', 0.0),
 ('PGM', 21.7),
 ('PIt2r', 5.2),
 ('ALCD2x', 0.0)]

In [36]:
ATPM = model.reactions.get_by_id("ATPM")
original_ATPM_lower_bound = ATPM.lower_bound

print(original_ATPM_lower_bound)

8.39


In [37]:
for reaction in model.reactions:

    if reaction.id in activity:

        value = activity[reaction.id]

        if reaction.reversibility:
            reaction.lower_bound = -value
            reaction.upper_bound = value

        else:
            reaction.upper_bound = value

In [38]:
model.reactions.get_by_id("ATPM").lower_bound = original_ATPM_lower_bound

In [39]:
model.reactions.get_by_id("ATPM")

Reaction identifier,ATPM
Name,ATP maintenance requirement
Memory address,0x261c0cd7750
Stoichiometry,atp_c + h2o_c --> adp_c + h_c + pi_c ATP C10H12N5O13P3 + H2O H2O --> ADP C10H12N5O10P2 + H+ + Phosphate
GPR,
Lower bound,8.39
Upper bound,1000.0


In [40]:
glc = model.reactions.get_by_id("EX_glc__D_e")

glc.lower_bound = -1000
glc.upper_bound = 1000

In [41]:
glc

Reaction identifier,EX_glc__D_e
Name,D-Glucose exchange
Memory address,0x261c0e410d0
Stoichiometry,glc__D_e <=> D-Glucose <=>
GPR,
Lower bound,-1000
Upper bound,1000


In [42]:
final_printout_table = pd.DataFrame({
    "Reaction": [r.id for r in model.reactions],
    "Lower bound": [r.lower_bound for r in model.reactions],
    "Upper bound": [r.upper_bound for r in model.reactions]
})

final_printout_table

,Reaction,Lower bound,Upper bound
0,PFK,0.0,13.1
1,PFL,0.0,0.0
2,PGI,-11.1,11.1
3,PGK,-24.0,24.0
4,PGL,0.0,7.3
...,...,...,...
90,NADH16,0.0,40.1
91,NADTRHD,0.0,1.3
92,NH4t,-1000.0,1000.0
93,O2t,-1000.0,1000.0


3a) 

In [43]:
# Reset glucose to the bounds from point 2 so this section can be rerun.
glc = model.reactions.get_by_id("EX_glc__D_e")
glc.lower_bound = -1000
glc.upper_bound = 1000

model.objective = "BIOMASS_Ecoli_core_w_GAM"
model.objective_direction = "max"
solution_3a = model.optimize()

print("solver status ", solution_3a.status)
print("max biomass production rate", round(solution_3a.objective_value, 6), "h^-1")
print("glucose exchange flux", round(solution_3a.fluxes["EX_glc__D_e"], 6), "mmol/gDW/h")

solver status  optimal
max biomass production rate 0.873286 h^-1
glucose exchange flux -10.620457 mmol/gDW/h


The maximal biomass production rate is about **0.873286 h^-1**. Glucose uptake has a high default bound here, so growth is limited by the reaction capacities and the other model constraints.

3b)
This limit means the cell can take up at most 5 mmol/gDW/h of glucose from its surroundings. We use -5 because glucose uptake is negative in this model. The limits from the expression data describe how much the enzymes can do. This new limit describes how much glucose is available to the cell.

In [44]:
glc.lower_bound = -5

print("Glucose exchange bounds:", glc.bounds)

Glucose exchange bounds: (-5, 1000)


 3c) 

In [45]:
solution_3c = model.optimize()

print("solver status", solution_3c.status)
print("max biomass production rate", round(solution_3a.objective_value, 6), "h^-1")
print("glucose exchange flux", round(solution_3a.fluxes["EX_glc__D_e"], 6), "mmol/gDW/h")

percentage_decrease = 100 * (solution_3a.objective_value - solution_3c.objective_value) / solution_3a.objective_value
print("Decrease in biomass production , round(percentage_decrease, 2), "%")

solver status optimal
max biomass production rate 0.873286 h^-1
glucose exchange flux -10.620457 mmol/gDW/h
Decrease in biomass production in % 52.41 %


The growth rate drops from 0.873286 to 0.415598 , so it is about 52.4% lower. In 3a the cell takes up about 10.62 mmol/gDW/h of glucose. Now it can only take up 5 and uses all of that allowance.

With less glucose the cell has less carbon and energy for growth. It still needs ATP for basic maintenance. The enzyme limits stay the same but the lower glucose supply means the cell cannot grow as fast.